# Test Butler from LSSTCam

In [43]:

import sys
import os
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
import numpy as np
import pandas as pd
from astropy.time import Time

In [41]:
from lsst.daf.butler import Butler

In [45]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"


In [ ]:
!eups list lsst_distrib

In [27]:
repo = "main"
instrument = "LSSTCam"
date_start = 20250415
date_selection = 20250416
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"
skymapName = "lsst_cells_v1"
COLLECTION_NAME = 'LSSTCam/raw/all' 

In [29]:
os.getenv("LSST_RESOURCES_WEBDAV_CONFIG")

'/pbs/home/d/dagoret/.lsst/webdav.yaml'

In [30]:
os.getenv("DAF_BUTLER_REPOSITORY_INDEX")

'/pbs/home/d/dagoret/.lsst/butler-repository-index.yaml'

## Butler init

In [32]:
#!eups list -s

In [35]:
butler = Butler(repo)
registry = butler.registry

In [34]:
butler.registry.queryDatasetTypes()

[DatasetType('skyMap', {skymap}, SkyMap),
 DatasetType('raw', {band, instrument, day_obs, detector, group, physical_filter, exposure}, Exposure),
 DatasetType('bfk', {instrument, detector}, BrighterFatterKernel, isCalibration=True),
 DatasetType('bias', {instrument, detector}, ExposureF, isCalibration=True),
 DatasetType('cti', {instrument, detector}, IsrCalib, isCalibration=True),
 DatasetType('dark', {instrument, detector}, ExposureF, isCalibration=True),
 DatasetType('defects', {instrument, detector}, Defects, isCalibration=True),
 DatasetType('linearizer', {instrument, detector}, Linearizer, isCalibration=True),
 DatasetType('ptc', {instrument, detector}, PhotonTransferCurveDataset, isCalibration=True),
 DatasetType('flat', {band, instrument, detector, physical_filter}, ExposureF, isCalibration=True),
 DatasetType('camera', {instrument}, Camera, isCalibration=True),
 DatasetType('crosstalk', {instrument, detector}, CrosstalkCalib, isCalibration=True),
 DatasetType('manual_defects',

In [36]:
all_collections = registry.queryCollections()

print(f"Nombre total de collections trouvées : {len(all_collections)}")

# Afficher les 20 premières pour un aperçu
print("\nLes 20 premières collections :")
for i, collection_name in enumerate(all_collections):
    print(collection_name)
    if i >= 19:
        break

Nombre total de collections trouvées : 165

Les 20 premières collections :
skymaps
LSSTCam/raw/all
LSSTCam/calib/DM-49175/run7/bfkGen.20250320a/20250324T220123Z
LSSTCam/calib/DM-49175/run7/biasGen.20250320a/20250325T205118Z
LSSTCam/calib/DM-49175/run7/ctiGen.20250320a/20250325T152139Z
LSSTCam/calib/DM-49175/run7/darkGen.20250320a/20250326T000943Z
LSSTCam/calib/DM-49175/run7/defectGen.20250401a/20250401T232630Z
LSSTCam/calib/DM-49175/run7/linearizerGen.20250320a/20250321T052032Z
LSSTCam/calib/DM-49175/run7/ptcGen.20250320a/20250323T161738Z
LSSTCam/calib/DM-49175/run7/bfk.20250320a
LSSTCam/calib/DM-49175/run7/bias.20250320a
LSSTCam/calib/DM-49175/run7/cti.20250320a
LSSTCam/calib/DM-49175/run7/dark.20250320a
LSSTCam/calib/DM-49175/run7/defects.20250401a
LSSTCam/calib/DM-49175/run7/linearizer.20250320a
LSSTCam/calib/DM-49175/run7/ptc.20250320a
LSSTCam/calib/DM-49679/pseudoFlats/pseudoFlatGen.20250401b/run
LSSTCam/calib/DM-49679/pseudoFlats/pseudoFlat.20250401b
LSSTCam/calib/DM-49832/curate

In [37]:
# Essayez la même méthode, mais sur l'objet 'registry'
try:
    exposure_records = registry.queryDimensionRecords('exposure')
    
    # Le reste de votre code
    list_of_records = list(exposure_records)
    print(f"Nombre total d'expositions trouvées : {len(list_of_records)}")

except AttributeError as e:
    print(f"Erreur de l'API Gen 3 : {e}")
    print("Veuillez vérifier l'initialisation de votre Butler, notamment l'URI et les collections.")

Nombre total d'expositions trouvées : 76324


In [38]:
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

In [44]:
# save the data array in rows before saving in pandas dataframe
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        jd_start = info.timespan.begin.value
        jd_end = info.timespan.end.value
        the_Time_start = Time(jd_start, format="jd", scale="utc")
        the_Time_end = Time(jd_end, format="jd", scale="utc")
        mjd_start = the_Time_start.mjd
        mjd_end = the_Time_end.mjd
        isot_start = the_Time_start.isot
        isot_end = the_Time_end.isot

        if count == 0:
            print("===== Time Conversion Debug Info =====")
            print(f"JD start      : {jd_start} (type: {type(jd_start)})")
            print(f"JD end        : {jd_end} (type: {type(jd_end)})")
            print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
            print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
            print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
            print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
            print("=======================================")

        # put row in a dictionnary before stacking
        row = {
            "id": info.id,
            "obs_id": info.obs_id,
            "day_obs": info.day_obs,
            "seq_num": info.seq_num,
            "time_start": isot_start,
            "time_end": isot_end,
            "type": info.observation_type,
            "target": info.target_name,
            "filter": info.physical_filter,
            "zenith_angle": info.zenith_angle,
            "expos": info.exposure_time,  # Exemple : adapter selon ton objet
            "ra": info.tracking_ra,
            "dec": info.tracking_dec,
            "skyangle": info.sky_angle,
            "azimuth": info.azimuth,
            "zenith": info.zenith_angle,
            "science_program": info.science_program,
            "jd": float(jd_start),
            "mjd": float(mjd_start),
        }
        rows.append(row)

    except ValueError as e:
        print(f"Erreur de valeur : {e}")
    except FileNotFoundError as e:
        print(f"Fichier introuvable : {e}")
    except Exception as e:
        print(f"Erreur inattendue : {type(e).__name__} - {e}")
        print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
        traceback.print_exc()  # affiche la stack trace complète


===== Time Conversion Debug Info =====
JD start      : 2460781.503444092 (type: <class 'numpy.float64'>)
JD end        : 2460781.5034443056 (type: <class 'numpy.float64'>)
MJD start     : 60781.00344409188 (type: <class 'numpy.float64'>)
MJD end       : 60781.00344430562 (type: <class 'numpy.float64'>)
ISOT start    : 2025-04-16T00:04:57.570 (type: <class 'str'>)
ISOT end      : 2025-04-16T00:04:57.588 (type: <class 'str'>)


In [46]:
# Création finale du DataFrame
df_exposure = pd.DataFrame(rows)

In [47]:
df_exposure

,id,obs_id,day_obs,seq_num,time_start,time_end,type,target,filter,zenith_angle,expos,ra,dec,skyangle,azimuth,zenith,science_program,jd,mjd
0,2025041500010,MC_O_20250415_000010,20250415,10,2025-04-16T00:04:57.570,2025-04-16T00:04:57.588,bias,AzEl,i_39,10.000000,0.0,123.124440,-28.757709,NaN,276.000000,10.000000,BLOCK-T446,2.460782e+06,60781.003444
1,2025041500018,MC_O_20250415_000018,20250415,18,2025-04-16T00:09:21.812,2025-04-16T00:09:21.831,bias,AzEl,i_39,9.999999,0.0,124.229133,-28.755571,NaN,276.000000,9.999999,BLOCK-T446,2.460782e+06,60781.006502
2,2025041500026,MC_O_20250415_000026,20250415,26,2025-04-16T00:13:45.808,2025-04-16T00:13:45.826,bias,AzEl,i_39,10.000001,0.0,125.331964,-28.754116,NaN,276.000000,10.000001,BLOCK-T446,2.460782e+06,60781.009558
3,2025041500034,MC_O_20250415_000034,20250415,34,2025-04-16T00:19:25.008,2025-04-16T00:19:25.025,bias,AzEl,i_39,10.000000,0.0,126.750552,-28.752909,NaN,276.000000,10.000000,BLOCK-T382,2.460782e+06,60781.013484
4,2025041500042,MC_O_20250415_000042,20250415,42,2025-04-16T00:25:49.267,2025-04-16T00:25:49.282,bias,Vela_SNR,i_39,17.051383,0.0,140.234750,-13.093580,NaN,2.184423,17.051383,BLOCK-T412,2.460782e+06,60781.017931
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73608,2025062000160,MC_O_20250620_000160,20250620,160,2025-06-21T00:21:45.331,2025-06-21T00:22:16.282,science,Prawn,r_57,40.607034,30.0,252.467325,-40.875583,191.139801,119.007474,40.607034,BLOCK-365,2.460848e+06,60847.015108
73609,2025062000161,MC_O_20250620_000161,20250620,161,2025-06-21T00:25:40.651,2025-06-21T00:26:11.600,science,Prawn,r_57,42.518470,30.0,255.954446,-40.406316,187.460314,118.379367,42.518470,BLOCK-365,2.460848e+06,60847.017832
73610,2025062000162,MC_O_20250620_000162,20250620,162,2025-06-21T00:26:18.946,2025-06-21T00:26:49.898,science,Prawn,r_57,39.283691,30.0,251.851521,-41.233104,189.959957,119.566895,39.283691,BLOCK-365,2.460848e+06,60847.018275
73611,2025062000163,MC_O_20250620_000163,20250620,163,2025-06-21T00:26:57.234,2025-06-21T00:27:28.187,science,Prawn,r_57,41.810671,30.0,255.358958,-40.769406,186.208895,118.878467,41.810671,BLOCK-365,2.460848e+06,60847.018718


In [48]:
df_exposure["target"].unique()

array(['AzEl', 'Vela_SNR', 'UNKNOWN', 'HD 106625', 'HD 123139', 'alf Cen',
       'Mirror covers operation', 'slew_icrs', 'azel_target',
       'Rubin_SV_225_-40 - filter change', 'Rubin_SV_225_-40', 'COSMOS',
       'Rubin_SV_216_-17', 'DEEP_A0', 'Carina', 'M49', 'Trifid-Lagoon',
       'Prawn', 'Rubin_SV_212_-7', 'DESI_SV3_R1', 'Virgo', 'WDFS1206-27',
       'Alpha Centauri', 'SF1615+001A - filter change', 'BLOCK-387_049',
       'BLOCK-387_031', 'BLOCK-387_025', 'BLOCK-387_027', 'Spica',
       'BLOCK-387_006', 'BLOCK-387_012', 'BLOCK-387_029', 'BLOCK-387_009',
       'BLOCK-387_015', 'BLOCK-387_026', 'BLOCK-387_037', 'BLOCK-387_002',
       'BLOCK-387_017', 'BLOCK-387_033', 'BLOCK-387_019', 'BLOCK-387_001',
       'BLOCK-387_011', 'BLOCK-387_021', 'BLOCK-387_053', 'BLOCK-387_013',
       'BLOCK-387_024', 'BLOCK-387_005', 'BLOCK-387_030', 'BLOCK-387_010',
       'BLOCK-387_052', 'BLOCK-387_023', 'BLOCK-387_038', 'BLOCK-387_014',
       'BLOCK-387_032', 'BLOCK-387_039', 'BLOCK-387_04

In [49]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)

In [50]:
df_science = df_science.copy()
df_science["band"] = df_science["filter"].apply(lambda x : x.split("_")[0])

In [51]:
df_science

,id,obs_id,day_obs,seq_num,time_start,time_end,type,target,filter,zenith_angle,expos,ra,dec,skyangle,azimuth,zenith,science_program,jd,mjd,band
0,2025041500139,MC_O_20250415_000139,20250415,139,2025-04-16T04:08:36.184,2025-04-16T04:08:39.133,science,HD 106625,i_39,22.940351,2.0,177.607603,-14.112866,80.000510,310.260029,22.940351,BLOCK-351,2.460782e+06,60781.172641,i
1,2025041500147,MC_O_20250415_000147,20250415,147,2025-04-16T04:36:47.941,2025-04-16T04:36:50.890,science,HD 106625,i_39,21.072834,2.0,183.952404,-17.541892,40.526647,302.354162,21.072834,BLOCK-351,2.460782e+06,60781.192222,i
2,2025041500155,MC_O_20250415_000155,20250415,155,2025-04-16T04:58:19.843,2025-04-16T04:58:22.791,science,HD 106625,i_39,25.801929,2.0,184.142134,-16.217836,40.526154,297.184388,25.801929,BLOCK-351,2.460782e+06,60781.207174,i
3,2025041500163,MC_O_20250415_000163,20250415,163,2025-04-16T05:14:58.806,2025-04-16T05:15:14.753,science,HD 106625,i_39,28.247698,15.0,185.086123,-16.358413,40.525751,292.986188,28.247698,BLOCK-351,2.460782e+06,60781.218736,i
4,2025041500140,MC_O_20250415_000140,20250415,140,2025-04-16T04:11:01.794,2025-04-16T04:11:04.736,science,HD 106625,i_39,16.725706,2.0,183.952462,-17.541894,80.001119,315.921652,16.725706,BLOCK-351,2.460782e+06,60781.174326,i
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22360,2025062000159,MC_O_20250620_000159,20250620,159,2025-06-21T00:21:07.698,2025-06-21T00:21:38.652,science,Prawn,r_57,41.984544,30.0,254.125332,-40.666385,190.856070,118.737998,41.984544,BLOCK-365,2.460848e+06,60847.014672,r
22361,2025062000160,MC_O_20250620_000160,20250620,160,2025-06-21T00:21:45.331,2025-06-21T00:22:16.282,science,Prawn,r_57,40.607034,30.0,252.467325,-40.875583,191.139801,119.007474,40.607034,BLOCK-365,2.460848e+06,60847.015108,r
22362,2025062000161,MC_O_20250620_000161,20250620,161,2025-06-21T00:25:40.651,2025-06-21T00:26:11.600,science,Prawn,r_57,42.518470,30.0,255.954446,-40.406316,187.460314,118.379367,42.518470,BLOCK-365,2.460848e+06,60847.017832,r
22363,2025062000162,MC_O_20250620_000162,20250620,162,2025-06-21T00:26:18.946,2025-06-21T00:26:49.898,science,Prawn,r_57,39.283691,30.0,251.851521,-41.233104,189.959957,119.566895,39.283691,BLOCK-365,2.460848e+06,60847.018275,r
